# Initialize the Prefect blocks

## Check prefect API URL

In [ ]:
import os

print(os.environ['PREFECT_API_URL'])

## Inspect the currect values

In [ ]:
import json
from prefect.blocks.system import Secret
print(json.dumps(Secret.load("env-vars", _sync=True).get(), indent=2))

## Update the values

In [ ]:
import os
from prefect.blocks.system import Secret

value = {
    # S3 bucket name and subfolder.
    # NOTE: the "share-bucket" block will be created automatically from these
    # variables. So if you change these variables, please also remove the
    # "share-bucket" block and it will be recreated.
    "PREFECT_BUCKET_NAME": "rs-dev-cluster-temp",
    "PREFECT_BUCKET_FOLDER": "prefect-share",

    # Token that was used to setup the Dask clusters.
    # See: https://gateway.dask.org/authentication.html#using-jupyterhub-s-authentication
    "JUPYTERHUB_API_TOKEN": "<your-token-value>",

    # Needed to run the performance indicator prefect flow
    # The values for the following fields should be taken from rs-infra-core inventory,
    # file rs-infra-core/inventory/sample/host_vars/setup/apps.yml.
    # There is a section named rs_performance_indicator. The values for the fields
    # are set at the cluster deployment. These values should be also used here
    # Here is the aforementioned section:
    # rs_performance_indicator:
    #  database:
    #    host: postgresql-cluster-rw.database.svc.cluster.local
    #    name: performance
    #    password: test
    #    username: test
    #    secret: pi-database-password
    "POSTGRES_HOST": "<cluster_postgres_host>", # default: "postgresql-cluster-rw.database.svc.cluster.local",
    "POSTGRES_USER": "<pi_postgres_user>",
    "POSTGRES_PASSWORD": "<pi_postgres_password>",
    "POSTGRES_PORT": "<cluster_postgres_port>", # normally, 5432
    "POSTGRES_PI_DB": "performance",
    # osam url, internal to the cluster
    "RSPY_HOST_OSAM": "http://rs-server-osam.processing.svc.cluster.local:8080",
}

# Jupyter env vars to pass to Prefect and Dask
for env in [
    "RSPY_UAC_CHECK_URL",
    "RSPY_WEBSITE",
    "TEMPO_ENDPOINT",
    "DASK_GATEWAY_PUBLIC",
    "DASK_GATEWAY_ADDRESS",
]:
    value[env] = os.environ[env]

# Save Prefect block
await Secret(value=value).save("env-vars", overwrite=True)